In [4]:
import os

os.chdir("..")

In [1]:
import json

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from mini_rag.bm25 import BM25
from mini_rag.dense import DenseSearch

c:\Users\vmaneata\Work\mini-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
with open("eval/eval_draft.jsonl", "r", encoding="utf-8") as f:
    eval_questions = [json.loads(line) for line in f]
with open("data/chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

In [6]:
bm25 = BM25(chunks, stem=False)
model = SentenceTransformer("all-MiniLM-L6-v2")
dense = DenseSearch(chunks, model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4568.28it/s]


In [11]:
def recall_at_k(df, k):
    found_at_k = df[(df["Prediction"] == "found") & (df["Rank"] <= k)][
        "Prediction"
    ].count()
    total = df["Prediction"].count()
    return found_at_k / total if total > 0 else 0

In [7]:
results_for_bm25 = []

for question in eval_questions:
    query = question["question"]
    source_id = question["source_id"]
    bm25_results = bm25.search(query, k=10)
    for index, (chunk, _) in enumerate(bm25_results):
        if source_id == chunk["id"]:
            results_for_bm25.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        results_for_bm25.append({"question": query, "Prediction": "missed", "Rank": None})

In [8]:
results_for_dense = []

for question in eval_questions:
    query = question["question"]
    source_id = question["source_id"]
    dense_results = dense.search(query, top_k=10)
    for index, (chunk, _) in enumerate(dense_results):
        if source_id == chunk["id"]:
            results_for_dense.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        results_for_dense.append({"question": query, "Prediction": "missed", "Rank": None})


In [ ]:
df_bm25 = pd.DataFrame(results_for_bm25)
df_dense = pd.DataFrame(results_for_dense)

In [13]:

df_dense.head()

,question,Prediction,Rank
0,Where can I find the auto-generated interactiv...,found,2.0
1,How can I stop FastAPI from sending exception ...,found,1.0
2,How can I securely store user passwords in a F...,found,3.0
3,"In programming terms, what are the different m...",found,1.0
4,Why do programs call operations like network r...,found,8.0


In [14]:
df_bm25.head()

,question,Prediction,Rank
0,Where can I find the auto-generated interactiv...,found,2.0
1,How can I stop FastAPI from sending exception ...,found,1.0
2,How can I securely store user passwords in a F...,found,1.0
3,"In programming terms, what are the different m...",found,1.0
4,Why do programs call operations like network r...,found,1.0


In [15]:
both = pd.merge(df_bm25, df_dense, on="question", suffixes=("_bm25", "_dense"))
both.head()

,question,Prediction_bm25,Rank_bm25,Prediction_dense,Rank_dense
0,Where can I find the auto-generated interactiv...,found,2.0,found,2.0
1,How can I stop FastAPI from sending exception ...,found,1.0,found,1.0
2,How can I securely store user passwords in a F...,found,1.0,found,3.0
3,"In programming terms, what are the different m...",found,1.0,found,1.0
4,Why do programs call operations like network r...,found,1.0,found,8.0


In [20]:
bm25_hit = both["Rank_bm25"] <= 10
dense_hit = both["Rank_dense"] <= 10

In [23]:
pd.crosstab(bm25_hit, dense_hit)

Rank_dense,False,True
Rank_bm25,,
False,6,4
True,3,29


In [24]:
from mini_rag.hybrid import HybridSearch

In [25]:
hybrid_search = HybridSearch(bm25, dense)

In [27]:
results_for_hybrid = []

for question in eval_questions:
    query = question["question"]
    source_id = question["source_id"]
    hybrid_results = hybrid_search.search(query, 10)
    for index, (chunk, _) in enumerate(hybrid_results):
        if source_id == chunk["id"]:
            results_for_hybrid.append(
                {"question": query, "Prediction": "found", "Rank": index + 1}
            )
            break
    else:
        results_for_hybrid.append({"question": query, "Prediction": "missed", "Rank": None})

In [28]:
df_hybrid = pd.DataFrame(results_for_hybrid)
df_hybrid.head()

,question,Prediction,Rank
0,Where can I find the auto-generated interactiv...,found,1.0
1,How can I stop FastAPI from sending exception ...,found,1.0
2,How can I securely store user passwords in a F...,found,1.0
3,"In programming terms, what are the different m...",found,1.0
4,Why do programs call operations like network r...,found,2.0


In [29]:
for k in range(1, 11):
    print(f"Hybrid Recall at {k}: {recall_at_k(df_hybrid, k)}")

Hybrid Recall at 1: 0.5952380952380952
Hybrid Recall at 2: 0.7142857142857143
Hybrid Recall at 3: 0.7380952380952381
Hybrid Recall at 4: 0.7619047619047619
Hybrid Recall at 5: 0.7857142857142857
Hybrid Recall at 6: 0.7857142857142857
Hybrid Recall at 7: 0.7857142857142857
Hybrid Recall at 8: 0.7857142857142857
Hybrid Recall at 9: 0.8571428571428571
Hybrid Recall at 10: 0.8571428571428571


In [30]:
mrr_hybrid = (1 / df_hybrid["Rank"]).fillna(0).mean()
print(f"Hybrid Mean Reciprocal Rank: {mrr_hybrid}")

Hybrid Mean Reciprocal Rank: 0.6813492063492064


In [31]:
ndcg10_hybrid = pd.Series(1 / np.log2(df_hybrid["Rank"] + 1)).fillna(0).mean()
print(f"Hybrid Normalized Discounted Cumulative Gain: {ndcg10_hybrid}")

Hybrid Normalized Discounted Cumulative Gain: 0.7232206695275516
